# DSE 230: PySpark Salary Linear Regression - Solution

## Linear regression exercise

--- 

PySpark API Documentation: https://spark.apache.org/docs/4.0.1/api/python/index.html


In [42]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pickle

%matplotlib inline

### Initialize Spark

In [43]:
# Suppress native-hadoop warning
import os; 
os.close(os.dup2(os.open(os.devnull, os.O_WRONLY), 2))
!sed -i '$a\# Add the line for suppressing the NativeCodeLoader warning \nlog4j.logger.org.apache.hadoop.util.NativeCodeLoader=ERROR,console' /$HADOOP_HOME/etc/hadoop/log4j.properties

In [44]:
import pyspark
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row

conf = pyspark.SparkConf().setAll([('spark.master', 'local[2]'),
                                   ('spark.app.name', 'Salary Linear Regression')])
spark = SparkSession.builder.config(conf=conf).getOrCreate()

print (pyspark.version.__version__)

4.1.1


### Read in data
Read in data from the local file system, using "file://\<path-to-file\>"

In [45]:
!pwd

/home/work/2026/S2-Spark


In [46]:
file_path = "file:///home/work/2026/S2-Spark/Salary_Data.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True).cache()

### EDA

#### Get number of rows

In [47]:
df.count()

30

#### Print schema

In [48]:
df.printSchema()

root
 |-- YearsExperience: double (nullable = true)
 |-- Salary: double (nullable = true)



#### Show first 10 rows

In [49]:
df.show(10)

+---------------+-------+
|YearsExperience| Salary|
+---------------+-------+
|            1.1|39343.0|
|            1.3|46205.0|
|            1.5|37731.0|
|            2.0|43525.0|
|            2.2|39891.0|
|            2.9|56642.0|
|            3.0|60150.0|
|            3.2|54445.0|
|            3.2|64445.0|
|            3.7|57189.0|
+---------------+-------+
only showing top 10 rows


#### Get summary statistics

In [50]:
df.describe().show()

+-------+------------------+------------------+
|summary|   YearsExperience|            Salary|
+-------+------------------+------------------+
|  count|                30|                30|
|   mean|5.3133333333333335|           76003.0|
| stddev| 2.837888157662718|27414.429784582302|
|    min|               1.1|           37731.0|
|    max|              10.5|          122391.0|
+-------+------------------+------------------+



### Data Prep

#### Rename columns
Rename column 'YearsExperience' to 'Input' and 'Salary' to 'Label', using <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumnRenamed.html#pyspark.sql.DataFrame.withColumnRenamed">withColumnRenamed</a>.
Then show the first 5 rows.

In [51]:
df = df.withColumnRenamed('YearsExperience', 'Input').withColumnRenamed('Salary', 'Label')
df.show(5)

+-----+-------+
|Input|  Label|
+-----+-------+
|  1.1|39343.0|
|  1.3|46205.0|
|  1.5|37731.0|
|  2.0|43525.0|
|  2.2|39891.0|
+-----+-------+
only showing top 5 rows


#### Assemble features into a feature vector

Assemble the input features (in this case, there is only 1 feature) into a feature vector.

In [52]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['Input'], outputCol='Features')
assembled_df = assembler.transform(df)

Show the first 5 rows for the resulting DF.

In [53]:
assembled_df.show(5)

+-----+-------+--------+
|Input|  Label|Features|
+-----+-------+--------+
|  1.1|39343.0|   [1.1]|
|  1.3|46205.0|   [1.3]|
|  1.5|37731.0|   [1.5]|
|  2.0|43525.0|   [2.0]|
|  2.2|39891.0|   [2.2]|
+-----+-------+--------+
only showing top 5 rows


#### Split data into train and test datasets

Randomly split the data into a train dataset and test dataset using randomSplit().  

Use only the Label column and assembled feature vector from the previous step.

Use 0.7 for train and 0.3 for test, with a seed of 135.

In [54]:
train_df, test_df = assembled_df.select(['Features', 'Label']) \
                                .randomSplit([0.7, 0.3], seed=135)

Print the count, first 5 rows, and summary statistics for the train dataset.

In [55]:
print("Train set size:", train_df.count())
train_df.show(5)
train_df.describe().show()

Train set size: 20
+--------+-------+
|Features|  Label|
+--------+-------+
|   [1.1]|39343.0|
|   [1.3]|46205.0|
|   [1.5]|37731.0|
|   [2.0]|43525.0|
|   [2.9]|56642.0|
+--------+-------+
only showing top 5 rows
+-------+------------------+
|summary|             Label|
+-------+------------------+
|  count|                20|
|   mean|           75169.9|
| stddev|26946.158651925456|
|    min|           37731.0|
|    max|          122391.0|
+-------+------------------+



Print the count, first 5 rows, and summary statistics for the test dataset.

In [56]:
print("Test set size:", test_df.count())
test_df.show(5)
test_df.describe().show()

Test set size: 10
+--------+-------+
|Features|  Label|
+--------+-------+
|   [2.2]|39891.0|
|   [3.0]|60150.0|
|   [3.2]|54445.0|
|   [3.7]|57189.0|
|   [3.9]|63218.0|
+--------+-------+
only showing top 5 rows
+-------+---------------+
|summary|          Label|
+-------+---------------+
|  count|             10|
|   mean|        77669.2|
| stddev|29734.978765383|
|    min|        39891.0|
|    max|       116969.0|
+-------+---------------+



#### Scale the features

Scale the features to ensure all features are on similar scales (with mean ~0 and stdev ~1), to improve convergence and stability.

In [57]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="Features", outputCol="ScaledFeatures",
    withMean=True, withStd=True)

scaler_model = scaler.fit(train_df)         

train_df = scaler_model.transform(train_df)  
test_df  = scaler_model.transform(test_df)  

train_df = train_df.select(['ScaledFeatures', 'Label'])
test_df  = test_df.select(['ScaledFeatures', 'Label'])

In [59]:
#checkout the scaled output
train_df.show(5)
test_df.show(5)

+--------------------+-------+
|      ScaledFeatures|  Label|
+--------------------+-------+
|[-1.428906385178194]|39343.0|
|[-1.3592885344995...|46205.0|
|[-1.289670683821001]|37731.0|
|[-1.1156260571245...|43525.0|
|[-0.8023457290708...|56642.0|
+--------------------+-------+
only showing top 5 rows
+--------------------+-------+
|      ScaledFeatures|  Label|
+--------------------+-------+
|[-1.0460082064459...|39891.0|
|[-0.7675368037315...|60150.0|
|[-0.6979189530529...|54445.0|
|[-0.5238743263564...|57189.0|
|[-0.4542564756778...|63218.0|
+--------------------+-------+
only showing top 5 rows


### Build Model

#### Build a linear regression model 
Build a linear regression model with the scaled feature vector as input and Label column as target.

After training the model, print out the model's coefficients and intercept.

In [60]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol = 'ScaledFeatures', labelCol='Label')
lr_model = lr.fit(train_df)

print("Coefficients: " + str(lr_model.coefficients))
print("Intercept: " + str(lr_model.intercept))

Coefficients: [26310.287927890728]
Intercept: 75169.89999999998


#### Print MAE (mean absolute error) from model summary

In [61]:
trainingSummary = lr_model.summary
print("MAE: %f" % trainingSummary.meanAbsoluteError)

MAE: 4784.408649


### Evaluate Model

#### Perform inference
Apply the model to train data.  
Then show the predictions, labels, and input feature vectors for the first 5 samples.

In [62]:
print("Predictions on Train Data:")
lr_train_predictions = lr_model.transform(train_df)
lr_train_predictions.show(5)

Predictions on Train Data:
+--------------------+-------+------------------+
|      ScaledFeatures|  Label|        prediction|
+--------------------+-------+------------------+
|[-1.428906385178194]|39343.0| 37574.96158396016|
|[-1.3592885344995...|46205.0| 39406.62728023494|
|[-1.289670683821001]|37731.0|41238.292976509714|
|[-1.1156260571245...|43525.0|45817.457217196665|
|[-0.8023457290708...|56642.0| 54059.95285043317|
+--------------------+-------+------------------+
only showing top 5 rows


Apply the model to test data.
Then show the predictions, labels, and input feature vectors for the first 5 samples.

In [63]:
print("Predictions on Test Data:")
lr_predictions = lr_model.transform(test_df)
lr_predictions.show(5)

Predictions on Test Data:
+--------------------+-------+------------------+
|      ScaledFeatures|  Label|        prediction|
+--------------------+-------+------------------+
|[-1.0460082064459...|39891.0|47649.122913471445|
|[-0.7675368037315...|60150.0| 54975.78569857056|
|[-0.6979189530529...|54445.0| 56807.45139484534|
|[-0.5238743263564...|57189.0| 61386.61563553228|
|[-0.4542564756778...|63218.0|63218.281331807055|
+--------------------+-------+------------------+
only showing top 5 rows


#### Calculate MAE (mean absolute error) on train and test data.

In [64]:
from pyspark.ml.evaluation import RegressionEvaluator

lr_evaluator = RegressionEvaluator(predictionCol="prediction", \
                 labelCol="Label",metricName="mae")

print("Mean absolute error on train data = %5.3f" % lr_evaluator.evaluate(lr_train_predictions))
print("Mean absolute error on test data = %5.3f" % lr_evaluator.evaluate(lr_predictions))

Mean absolute error on train data = 4784.409
Mean absolute error on test data = 4537.478


### Stop Spark Session

In [65]:
spark.stop()